# TSCT — Générateur de QCM (v1.0)

Colab wrapper around `scripts/run_qcm.py` from the public repo **fcavarretta/instest** — no logic lives here. Stdlib only, nothing to install.

**One-time setup** (🔑 Secrets panel, left sidebar — enable notebook access for each): `GEMINI_API_KEY` (required), and `GITHUB_TOKEN` (fine-grained token on `fcavarretta/instest` with **Contents: Read and write**) — **only needed to ⬆ PUSH in-class edits back**; reading/opening needs nothing, the repo is public.

**In class**: *Runtime → Run all*, then run the session cell pair you need. The repo is **⬇ pulled** into `/content/instest` (Setup section) — a full git working copy: browse and edit any YAML or prompt via the 📁 Files panel. **If you edited something, run the ⬆ PUSH cell at the bottom before leaving** — the Colab machine's disk is wiped when the runtime ends, and only pushed edits survive.

(The notebook itself is covered too: auto-push and ⬆ PUSH snapshot the live document into the repo before sending — *File → Save a copy in GitHub* is no longer needed.)

## 1 · Setup — *Run all* passes through here; collapse afterwards (▸ next to this title)

In [ ]:
# @title Mount Google Drive (audio in, results out) { display-mode: "form" }
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title ⬇ PULL — get the latest from GitHub (start of session; never clobbers unpushed edits) { display-mode: "form" }
import os, subprocess

REPO = 'github.com/fcavarretta/instest.git'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
# Public repo: reading works without a token; the token (when present) enables ⬆ PUSH.
url = f'https://{token}@{REPO}' if token else f'https://{REPO}'

def git(*args):
    return subprocess.run(['git', '-C', '/content/instest', *args], capture_output=True, text=True)

def show(r):
    out = r.stdout + r.stderr
    # Never print raw git output without masking — it can contain the tokenized URL.
    print(out.replace(token, '***') if token else out)
    assert r.returncode == 0, 'git failed — see output above'

if not os.path.exists('/content/instest'):
    show(subprocess.run(['git', 'clone', url, '/content/instest'], capture_output=True, text=True))
else:
    # Keep the remote in sync with the current token: granting the secret later
    # then re-running THIS cell is enough to make ⬆ PUSH work.
    git('remote', 'set-url', 'origin', url)
    if git('status', '--porcelain').stdout.strip():
        print('⚠️ Local edits not yet pushed to GitHub — skipping the pull so nothing is lost.')
        print('   Run the ⬆ PUSH cell at the bottom, then re-run this cell.')
    else:
        show(git('pull', '--ff-only'))

# Capture the CURRENT state of this notebook document into the repo clone, so
# auto-push / ⬆ PUSH send notebook edits too (outputs stripped; Colab-internal API).
def sync_notebook_into_repo():
    try:
        from google.colab import _message
        import json as _json
        resp = _message.blocking_request('get_ipynb', request='', timeout_sec=30)
        data = (resp or {}).get('ipynb')
        if not data:
            return False
        for cell in data.get('cells', []):
            if cell.get('cell_type') == 'code':
                cell['outputs'] = []
                cell['execution_count'] = None
            # Colab stamps volatile run metadata into cells (executionInfo timestamps,
            # outputId) — stripping it stops phantom 'changes' on every tick.
            for k in ('executionInfo', 'outputId'):
                cell.get('metadata', {}).pop(k, None)
        with open('/content/instest/notebook/run_qcm.ipynb', 'w') as f:
            _json.dump(data, f, indent=1, ensure_ascii=False)
        return True
    except Exception:
        return False


In [ ]:
# @title ⬆ Auto-PUSH — background send to GitHub every N min (live status below) { display-mode: "form" }
AUTOSAVE_MINUTES = 2  # @param {type:"integer"}
# Status updates at every check: ✅ + time of the last successful push, ⚠️ on any
# problem (including a token that CANNOT push — probed at arm time). Each check
# first snapshots THIS notebook document into the repo, so notebook edits are
# pushed too. Re-running this cell supersedes the previous loop.
import threading, time, datetime
import ipywidgets as widgets
from IPython.display import display

_status = widgets.HTML(value='⏳ Auto-push arming…')
display(_status)
_autosave_state = {'last_push': None, 'ok': True}
_AUTOSAVE_GEN = globals().get('_AUTOSAVE_GEN', 0) + 1

def _tick():
    now = datetime.datetime.now().strftime('%H:%M')
    sync_notebook_into_repo()
    if git('status', '--porcelain').stdout.strip():
        git('add', '-A')
        git('commit', '-m', f'Auto-save from Colab, {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}')
    # Push whenever local commits are ahead — also retries an earlier failed push.
    ahead = git('rev-list', '--count', '@{u}..HEAD').stdout.strip()
    if ahead not in ('', '0'):
        if git('push').returncode == 0:
            _autosave_state.update(last_push=now, ok=True)
            note = 'pushed'
        else:
            _autosave_state['ok'] = False
            note = 'PUSH FAILED — run the ⬆ PUSH cell'
    else:
        note = 'nothing to send'
    icon = '✅' if _autosave_state['ok'] else '⚠️'
    push_txt = f"last push <b>{_autosave_state['last_push']}</b>" if _autosave_state['last_push'] else 'no push yet'
    _status.value = f"{icon} {push_txt} · checked {now} ({note}) · every {AUTOSAVE_MINUTES} min"

git('config', 'user.name', 'Fabrice Cavarretta (Colab)')
git('config', 'user.email', 'fabrice@cavarretta.fr')
_probe = git('push', '--dry-run')
if _probe.returncode != 0:
    _autosave_state['ok'] = False
    _msg = ('⚠️ PUSH NOT POSSIBLE — after a runtime restart, run ⬇ PULL first (it re-points the '
            'remote with the token). Otherwise: token missing, expired, or not Read-and-write on instest.')
    _status.value = _msg
    print(_msg)
    _detail = (_probe.stdout + _probe.stderr).strip()
    if token:
        _detail = _detail.replace(token, '***')
    print('git said:', _detail[-300:] or '(no detail)')
else:
    _tick()  # immediate synchronous first check — result visible right away
    print('Auto-push armed. Status:', _status.value.replace('<b>', '').replace('</b>', ''))

def _autosave_loop(gen):
    while True:
        time.sleep(AUTOSAVE_MINUTES * 60)
        if gen != _AUTOSAVE_GEN:
            return  # a newer arm of this cell took over
        try:
            _tick()
        except Exception as e:
            _autosave_state['ok'] = False
            _status.value = f'⚠️ auto-push error: {e} — run the ⬆ PUSH cell'

threading.Thread(target=_autosave_loop, args=(_AUTOSAVE_GEN,), name=f'tsct-autosave-{_AUTOSAVE_GEN}', daemon=True).start()

In [ ]:
# @title 🩺 MODELS CHECK — key + quota canary (run before class) { display-mode: "form" }
# Pings each configured model once (~fractions of a cent). Proves: key valid,
# model reachable, quota NOT exhausted — right now. True billing balance is not
# queryable via an API key: set a Google Cloud budget alert for advance warning.
import sys, time, yaml
sys.path.insert(0, '/content/instest/scripts')
from lib.io_layer import get_api_key
from lib.config import CallSettings
from lib.gemini_client import call_model, ApiError

_models = yaml.safe_load(open('/content/instest/resources/system.yaml'))['models']
_by_model = {}
for _role, _m in _models.items():
    _by_model.setdefault(_m, []).append(_role)
_key = get_api_key()
for _m, _roles in _by_model.items():
    _t0 = time.time()
    try:
        _text, _usage = call_model(_m, [{"text": "Reply with: ok"}],
                                   CallSettings(max_output_tokens=64, temperature=0.0, thinking_level='low'),
                                   _key, call_name='check', fast_fail=True)
        print(f"✅ {_m} ({'+'.join(_roles)}): responds in {time.time()-_t0:.1f}s → {_usage.resolved_model}")
    except ApiError as _e:
        _s = str(_e)
        if '429' in _s or 'RESOURCE_EXHAUSTED' in _s.upper():
            print(f"⛔ {_m} ({'+'.join(_roles)}): QUOTA/BUDGET EXHAUSTED — do NOT count on it in class. {_s[:200]}")
        else:
            print(f"⚠️ {_m} ({'+'.join(_roles)}): {_s[:250]}")

In [ ]:
# @title Course — which course.yaml applies to the sessions below { display-mode: "form" }
COURSE = '/content/drive/MyDrive/_TSCT/course.yaml'  # @param {type:"string"}
print(f'Course file: {COURSE}')

## 2 · Sessions — one cell per session; run the one that just ended

In [ ]:
# @title Session 1 · parameters — run me first (defines AUDIO/SESSION for both steps) { display-mode: "form" }
AUDIO = '/content/drive/MyDrive/_TSCT/11-05.m4a'  # @param {type:"string"}
SESSION = '/content/drive/MyDrive/_TSCT/session.yaml'  # @param {type:"string"}
OUTPUT = ''  # optional folder override; empty = outputs beside the audio (hidden plumbing)

import sys
sys.path.insert(0, '/content/instest/scripts')
print(f'Parameters set: AUDIO={AUDIO}')

In [ ]:
# @title Session 1 · TRANSCRIBE (audio → X.transcript.md — costly; below the STOP on purpose) { display-mode: "form" }
HALT_ON_FAILURE = True  # @param {type:"boolean"}
# Needs the parameters cell run first. Review the transcript if you wish, then run step 2.
from run_qcm import main
args = [SESSION, '--audio', AUDIO, '--transcribe-only', '--course', COURSE]
if OUTPUT.strip():
    args += ['--output-root', OUTPUT]
rc = main(args)
# The halt is deliberately the LAST statement: partial files, metadata and the
# banner are already written/printed above — nothing is skipped by stopping.
if rc and HALT_ON_FAILURE:
    raise SystemExit(f'⛔ STEP FAILED (rc={rc}) — READ THE OUTPUT ABOVE. Halted; run the next cell manually to proceed.')
elif rc:
    print(f'⚠️ STEP FAILED (rc={rc}) — continuing because HALT_ON_FAILURE is unticked.')

In [ ]:
# @title Session 1 · GENERATE (existing transcript → X.questions.gift) { display-mode: "form" }
HALT_ON_FAILURE = True  # @param {type:"boolean"}
# Needs the parameters cell run first (transcribe not required if a transcript already exists).
from pathlib import Path
from run_qcm import main
from lib.runfolder import find_latest_transcript

transcript = find_latest_transcript(Path(AUDIO), Path(OUTPUT) if OUTPUT.strip() else None)
print(f'Using transcript: {transcript}')
args = [SESSION, '--generate-only', '--transcript', str(transcript), '--course', COURSE]
if OUTPUT.strip():
    args += ['--output-root', OUTPUT]
rc = main(args)
# The halt is deliberately the LAST statement: partial files, metadata and the
# banner are already written/printed above — nothing is skipped by stopping.
if rc and HALT_ON_FAILURE:
    raise SystemExit(f'⛔ STEP FAILED (rc={rc}) — READ THE OUTPUT ABOVE. Halted; run the next cell manually to proceed.')
elif rc:
    print(f'⚠️ STEP FAILED (rc={rc}) — continuing because HALT_ON_FAILURE is unticked.')

## 3 · ⬆ PUSH — send your work home before leaving (auto-push is only the net)

In [ ]:
# @title ⬆ PUSH — send your edits (incl. this notebook) to GitHub (run before leaving class) { display-mode: "form" }
import datetime
git('config', 'user.name', 'Fabrice Cavarretta (Colab)')
git('config', 'user.email', 'fabrice@cavarretta.fr')
sync_notebook_into_repo()  # snapshot the live notebook document into the clone
if not git('status', '--porcelain').stdout.strip():
    print('Nothing to send — working copy is clean.')
else:
    git('add', '-A')
    stamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    show(git('commit', '-m', f'In-class edits from Colab, {stamp}'))
    r = git('push')
    if r.returncode != 0:
        # Remote moved since we cloned (e.g. a push from home) — replay our commit on top.
        print('Push rejected — pulling remote changes and retrying…')
        p = git('pull', '--rebase')
        if p.returncode != 0:
            git('rebase', '--abort')
            print('❌ Same file changed on both sides — nothing lost (commit is local); copy this output to the AI.')
            print((p.stdout + p.stderr).replace(token, '***') if token else (p.stdout + p.stderr))
        else:
            r = git('push')
    if r.returncode == 0:
        print('✅ Pushed to GitHub — safe to close.')
    else:
        out = (r.stdout + r.stderr)
        print('❌ push failed:', out.replace(token, '***') if token else out)

In [ ]:
# @title ⏹ STOP — Run all halts here; cells below run only manually { display-mode: "form" }
raise SystemExit('⏹ Stopped on purpose — TRANSCRIBE (below) runs only when you launch it yourself.')